# multigauge: validate the pre-trained models on real basins (Kaggle)

Uses the models trained by **multigauge pretraining and validation** (attach that notebook's output with
*Add Input → Your Work*), so nothing is re-trained from scratch. Runs the gauge-value study on five large
U.S. basins the model never saw, plus sub-basin and forecast tests for the Washita.

**Settings:** Accelerator GPU, Internet on, then *Save Version → Save & Run All*.

In [ ]:
import torch, shutil
from pathlib import Path
print("GPU available:", torch.cuda.is_available())
WORK = Path("/kaggle/working/multigauge"); WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
!rm -rf /kaggle/temp/multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git /kaggle/temp/multigauge-hydro
!pip install -q -e "/kaggle/temp/multigauge-hydro[download]"
import sys; sys.path.insert(0, "/kaggle/temp/multigauge-hydro/src")
REPO = Path("/kaggle/temp/multigauge-hydro")

In [ ]:
# bring in the trained models (and any basin data already downloaded) from the attached output
found = sorted(Path("/kaggle/input").rglob("camels-us/model.json"))
assert found, "Attach the output of 'multigauge pretraining and validation' (Add Input -> Your Work)."
prev = found[0].parent.parent.parent          # .../multigauge
for sub in ("pretrained", "validation"):
    if (prev / sub).exists():
        shutil.copytree(prev / sub, WORK / sub, dirs_exist_ok=True)
for f in WORK.rglob("run_info.json"):          # earlier failed attempt left no results; keep data only
    pass
print(sorted(p.name for p in (WORK / "pretrained").iterdir()))

In [ ]:
import numpy as np, pandas as pd, multigauge as mg
from multigauge.nn import StreamflowModel
from multigauge.hybrid import HybridModel
model = StreamflowModel.load(WORK / "pretrained" / "camels-us")
hyb = HybridModel.load(WORK / "pretrained" / "camels-us-dhbv") if (WORK / "pretrained" / "camels-us-dhbv" / "hybrid.json").exists() else None
print("LSTM best validation NSE:", round(model.info.get("best_val_score", float("nan")), 3), "| hybrid:", hyb is not None)

## Validate on five basins the model never saw

In [ ]:
import traceback
from multigauge.prepare import prepare_usgs_basin
from multigauge.study import StudyConfig, load_saved_study
from multigauge.report import summary_table
SITES = {
    "07326500": "Washita River at Anadarko, OK",
    "05453100": "Iowa River at Marengo, IA",
    "02347500": "Flint River, GA",
    "08167000": "Guadalupe River at Comfort, TX",
    "01643000": "Monocacy River at Jug Bridge, MD",
}
START, END = "2000-10-01", "2023-09-30"
PERIODS = dict(train_period=("2001-10-01", "2013-09-30"), validation_period=("2013-10-01", "2017-09-30"),
               test_period=("2017-10-01", "2023-09-30"))
OUT = WORK / "validation"; OUT.mkdir(exist_ok=True)
for site, label in SITES.items():
    if (OUT / site / "results" / "run_info.json").exists():
        print(f"[{site}] already done"); continue
    try:
        if (OUT / site / "data" / "basin.json").exists():
            d = mg.BasinData.from_folder(OUT / site / "data")
        else:
            d = prepare_usgs_basin(site, START, END, OUT / site / "data", name=label)
        res = mg.run_study(d, model, StudyConfig(n_seeds=3, finetune_epochs=20, **PERIODS), hybrid=hyb,
                           progress=lambda f, m: print(f"\r[{site}] {int(f*100):3d}% {m[:70]:<70}", end=""))
        print()
        out = res.save(OUT / site / "results"); mg.write_report(res, d, out / "report.html")
        print(summary_table(res).round(3)[["NSE", "KGE", "CRPS"]].to_string(), "\n")
    except Exception as exc:
        print(f"\n[{site}] FAILED: {exc}"); traceback.print_exc()

## Washita: sub-basins and forecasts

In [ ]:
import json
from shapely.geometry import mapping
from multigauge.routing import SubbasinSet, auto_zones, run_routed_study
from multigauge.forecast import ForecastConfig, ForecastModel
from multigauge.download import daymet_basin
from multigauge.study import make_periods
site = "07326500"; R = OUT / site / "results"
if (R / "run_info.json").exists() and not (R / "forecast_skill.csv").exists():
    d = mg.BasinData.from_folder(OUT / site / "data")
    zdir = OUT / site / "data" / "subbasins"; zdir.mkdir(exist_ok=True)
    zones = auto_zones(d.boundary, d.outlet, 4); feats = []
    for i, z in enumerate(zones):
        zid = f"Z{i+1}"
        if not (zdir / f"{zid}.csv").exists():
            daymet_basin(z, START, END, n_points=9).to_csv(zdir / f"{zid}.csv", date_format="%Y-%m-%d")
        feats.append({"type": "Feature", "geometry": mapping(z), "properties": {"id": zid, "downstream": None if i == 0 else f"Z{i}"}})
    (zdir / "subbasins.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": feats}))
    periods = make_periods(d.dates, StudyConfig(**PERIODS), model.cfg.lookback)
    routed = run_routed_study(d, model, SubbasinSet.from_folder(zdir), periods, routing="gamma", epochs=10)
    routed.summary.to_csv(R / "subbasin_models.csv"); routed.contributions.to_csv(R / "subbasin_contributions.csv")
    print(routed.summary.xs("test", level="period")[["NSE", "KGE"]].round(3))
    b = d.to_basin(); m = {k: np.asarray((d.dates >= a) & (d.dates <= e)) for k, (a, e) in periods.items()}
    ft = mg.finetune(model, b, m["train"], m["validation"], epochs=20)
    fm = ForecastModel(ft, ForecastConfig(epochs=15)).fit(b, m["train"], m["validation"])
    iss = np.where(m["test"])[0]; iss = iss[iss < len(b.dates) - 8]
    skill = fm.skill_by_lead(b, iss); skill.to_csv(R / "forecast_skill.csv", index=False)
    print(skill.round(3).to_string(index=False))

## Summary across basins

In [ ]:
rows = []
for site in SITES:
    if not (OUT / site / "results" / "run_info.json").exists(): continue
    res = load_saved_study(OUT / site / "results")
    s = res.summary.xs("test", level="period")
    gv = res.gauge_value[res.gauge_value["period"] == "test"].set_index("gauge_input")
    r = {"site": site, "basin": res.basin_name, "area_km2": round(res.area_km2), "gauges": len(res.station_summary)}
    for k in ["gr4j", "local", "pretrained", "finetuned", "ft_voronoi", "ft_disaggregated", "dhbv", "ensemble"]:
        if k in s.index: r[f"NSE_{k}"] = round(s.loc[k, "NSE"], 3)
    best = gv["NSE_gain"].idxmax()
    r.update(best_gauge_input=best, NSE_gain_from_gauges=round(gv.loc[best, "NSE_gain"], 3),
             CRPS_reduction_pct=round(gv.loc[best, "CRPS_reduction_pct"], 1))
    rows.append(r)
summary = pd.DataFrame(rows); summary.to_csv(OUT / "summary.csv", index=False); summary

## Package the results

In [ ]:
!cd /kaggle/working && rm -f multigauge_results.zip && zip -qr multigauge_results.zip multigauge -x "*checkpoint*" && ls -la multigauge_results.zip